## Multilayer perceptron

### data preparation

In [146]:
import pandas as pd
import numpy as np
import keras
import tensorflow as tf


data_dir = "/Users/matthew/Documents/Rutgers/25 fall/ML/ml_project/data/merged_all.parquet"
data = pd.read_parquet(data_dir)

data.head()

,date,cusip_x,company_symbol,tmt,coupon,t_spread,yield,ret_eom,rating_A,rating_AA,...,sp500_ret,ir3m_chg,ir10y_chg,vix_chg,gdp_gr,cpi_infl,sector_etf,etf_price,etf_return,month_y
0,2002-07-31,000361AB1,AIR,1.23,0.0725,NaN,NaN,NaN,NaN,NaN,...,-0.072455,-0.046,-0.219,0.271271,0.000000,0.000557,XLI,15.428077,-0.061331,2002-07
1,2002-08-31,000361AB1,AIR,1.14,0.0725,NaN,0.04827,0.008709,0.0,0.0,...,-0.079004,0.006,-0.359,0.261024,0.004065,0.002227,XLI,14.554782,-0.056604,2002-08
2,2002-09-30,000361AB1,AIR,1.06,0.0725,NaN,0.04386,0.006141,0.0,0.0,...,0.004881,-0.020,-0.328,0.019045,0.000000,0.002778,XLI,14.328375,-0.015556,2002-09
3,2002-10-31,000361AB1,AIR,0.97,0.0725,NaN,0.04122,0.005690,0.0,0.0,...,-0.110024,-0.118,-0.530,0.215993,0.000000,0.001662,XLI,12.674906,-0.115398,2002-10
4,2002-11-30,000361AB1,AIR,0.89,0.0725,NaN,0.03873,0.001961,0.0,0.0,...,0.086449,-0.110,0.304,-0.215419,0.001236,0.002212,XLI,13.298265,0.049181,2002-11


In [147]:
data.columns

Index(['date', 'cusip_x', 'company_symbol', 'tmt', 'coupon', 't_spread',
       'yield', 'ret_eom', 'rating_A', 'rating_AA', 'rating_AAA', 'rating_B',
       'rating_BB', 'rating_BBB', 'rating_C', 'rating_CC', 'rating_CCC',
       'rating_D', 'upgrade', 'downgrade', 'gs3m', 'term_spread', 'issuer6',
       'month_x', 'PERMNO', 'ret', 'vol', 'dvol', 'turnover', 'mktcap',
       'bidask', 'numtrades', 'price_mean', 'GVKEY', 'atq', 'ceqq', 'cshoq',
       'ltq', 'niq', 'oibdpq', 'revtq', 'xintq', 'prccq', 'cusip_y', 'costat',
       'gsector', 'log_atq', 'lev_total', 'equity_ratio', 'roa',
       'profit_margin', 'int_coverage', 'mkt_cap', 'market_to_book',
       'atq_growth', 'revtq_growth', 'niq_growth', 'sp500_ret', 'ir3m_chg',
       'ir10y_chg', 'vix_chg', 'gdp_gr', 'cpi_infl', 'sector_etf', 'etf_price',
       'etf_return', 'month_y'],
      dtype='object')

In [148]:
data['date'] = pd.to_datetime(data['date'])
data = data.sort_values(['cusip_x','date'])
data['ytm_change'] = data.groupby('cusip_x')['yield'].diff()
data = data.dropna(subset=['ytm_change'])

In [149]:
exclude_cols = [
    'date','cusip_x','company_symbol','issuer6','PERMNO','GVKEY','cusip_y',
    'rating_AA','rating_BBB','rating_B', 'rating_C', 'rating_CC',
     'rating_D', 'rating_CCC','rating_A','rating_AAA', 'rating_BB', 'gsector', 'month_y',
    'upgrade','downgrade','month_x','ytm_change'
]

numeric_cols = [
    col for col in data.columns
    if col not in exclude_cols and data[col].dtype != 'object'
]

print(f"We standarlize these lines:\n{numeric_cols}")

We standarlize these lines:
['tmt', 'coupon', 't_spread', 'yield', 'ret_eom', 'gs3m', 'term_spread', 'ret', 'vol', 'dvol', 'turnover', 'mktcap', 'bidask', 'numtrades', 'price_mean', 'atq', 'ceqq', 'cshoq', 'ltq', 'niq', 'oibdpq', 'revtq', 'xintq', 'prccq', 'log_atq', 'lev_total', 'equity_ratio', 'roa', 'profit_margin', 'int_coverage', 'mkt_cap', 'market_to_book', 'atq_growth', 'revtq_growth', 'niq_growth', 'sp500_ret', 'ir3m_chg', 'ir10y_chg', 'vix_chg', 'gdp_gr', 'cpi_infl', 'etf_price', 'etf_return']


In [150]:
data_clean = data.copy()

data_clean[numeric_cols] = data_clean[numeric_cols].replace([np.inf, -np.inf], np.nan)

data_clean = data_clean.dropna(subset=numeric_cols)

### modeling

In [151]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

train_df, test_df = train_test_split(
    data_clean, test_size=0.2, random_state=1, shuffle=False
)

train_numeric = train_df[numeric_cols].values
test_numeric  = test_df[numeric_cols].values

scaler = StandardScaler()
train_numeric = scaler.fit_transform(train_numeric)
test_numeric  = scaler.transform(test_numeric)

max_z = 10.0
train_numeric = np.clip(train_numeric, -max_z, max_z)
test_numeric  = np.clip(test_numeric,  -max_z, max_z)

x_train = train_numeric
x_test  = test_numeric

y_train = train_df['ytm_change'].values
y_test  = test_df['ytm_change'].values

print("x_train mean/std:", np.mean(x_train), np.std(x_train))
print("x_test  mean/std:", np.mean(x_test),  np.std(x_test))

x_train mean/std: -0.0023567650758617126 0.8892652881864406
x_test  mean/std: -0.03167658134536295 0.7956628107263947


In [152]:
input_dim = x_train.shape[1]
inputs = keras.Input(shape=(input_dim,))

d = keras.layers.Dense(units=8, activation='relu')(inputs)
d = keras.layers.Dense(units=8, activation='relu')(d)
d = keras.layers.Dense(units=8, activation='relu')(d)
outputs = keras.layers.Dense(units=1)(d)

model1 = keras.Model(inputs=inputs, outputs=outputs)
model1.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 43)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 8)              │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 505 (1.97 KB)

 Trainable params: 505 (1.97 KB)

 Non-trainable params: 0 (0.00 B)

In [153]:
print(np.isinf(x_train).sum())
print(np.isinf(x_test).sum())
print(np.isinf(y_train).sum())
print(np.isinf(y_test).sum())

0
0
0
0


In [154]:
print(np.isnan(x_train).sum())
print(np.isnan(x_test).sum())
print(np.isnan(y_train).sum())
print(np.isnan(y_test).sum())

0
0
0
0


In [155]:
optimizer = keras.optimizers.Adam(
    learning_rate=1e-4,
    clipnorm=1.0
)

model1.compile(
    loss='mse',
    metrics=['mse', 'mae'],
    optimizer=optimizer
)

early_stopping_cb = keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

history = model1.fit(
    x=x_train,
    y=y_train,
    batch_size=64,
    epochs=200,
    shuffle=False,
    validation_split=0.2,
    callbacks=[early_stopping_cb],
    verbose=2
)


Epoch 1/200
5854/5854 - 2s - 389us/step - loss: 0.0043 - mae: 0.0305 - mse: 0.0043 - val_loss: 4.3193e-04 - val_mae: 0.0090 - val_mse: 4.3193e-04
Epoch 2/200
5854/5854 - 2s - 326us/step - loss: 2.5912e-04 - mae: 0.0057 - mse: 2.5912e-04 - val_loss: 2.3781e-04 - val_mae: 0.0048 - val_mse: 2.3781e-04
Epoch 3/200
5854/5854 - 2s - 325us/step - loss: 1.3365e-04 - mae: 0.0040 - mse: 1.3365e-04 - val_loss: 2.2324e-04 - val_mae: 0.0044 - val_mse: 2.2324e-04
Epoch 4/200
5854/5854 - 2s - 325us/step - loss: 1.1909e-04 - mae: 0.0037 - mse: 1.1909e-04 - val_loss: 2.1291e-04 - val_mae: 0.0041 - val_mse: 2.1291e-04
Epoch 5/200
5854/5854 - 2s - 325us/step - loss: 1.1411e-04 - mae: 0.0037 - mse: 1.1411e-04 - val_loss: 2.0766e-04 - val_mae: 0.0040 - val_mse: 2.0766e-04
Epoch 6/200
5854/5854 - 2s - 325us/step - loss: 1.0947e-04 - mae: 0.0036 - mse: 1.0947e-04 - val_loss: 2.0385e-04 - val_mae: 0.0039 - val_mse: 2.0385e-04
Epoch 7/200
5854/5854 - 2s - 325us/step - loss: 1.0616e-04 - mae: 0.0036 - mse: 1.06

In [156]:
loss_test, mse_test, mae_test = model1.evaluate(x_test, y_test, verbose=0)

print(loss_test, mse_test, mae_test)


9.066806524060667e-05 9.066806524060667e-05 0.0032209649216383696


In [157]:
model1.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 43)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 8)              │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,517 (5.93 KB)

 Trainable params: 505 (1.97 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,012 (3.96 KB)